In [1]:
# ====================== GPU PICKER (server-friendly) ======================
# Set which GPU to use (0, 1, ...). Set to None to NOT force anything (respects the environment).
GPU_ID = 1 # Change to None when you want to leave it free for other users
import os
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
if GPU_ID is not None:
    os.environ["CUDA_VISIBLE_DEVICES"] = str(GPU_ID)
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
import torch
torch.set_num_threads(1)
torch.set_num_interop_threads(1)
torch.backends.cuda.matmul.allow_tf32 = True
try:
    torch.backends.cudnn.allow_tf32 = True
except Exception:
    pass
torch.backends.cudnn.benchmark = True  # if the size varies A LOT, consider False
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    torch.cuda.set_device(0)
print("Device:", device, "| Visible devices (after mapping):", torch.cuda.device_count())
print("CUDA_VISIBLE_DEVICES =", os.environ.get("CUDA_VISIBLE_DEVICES"))
if device == "cuda":
    try:
        print("Using mapped GPU 0:", torch.cuda.get_device_name(0))
    except Exception:
        pass
    for i in range(torch.cuda.device_count()):
        try:
            print(f"Device {i}:", torch.cuda.get_device_name(i))
        except Exception:
            pass
import sys
import time
from datetime import datetime

Device: cuda | Visible devices (after mapping): 1
CUDA_VISIBLE_DEVICES = 1
Using mapped GPU 0: NVIDIA L40S
Device 0: NVIDIA L40S


In [2]:
import sys
sys.path.insert(0, "/workspace/app")
import matplotlib.pyplot as plt
import pandas as pd
import shap
import numpy as np
import os
import importlib
from coding.Data_Processing import Run_CNN as rf
importlib.reload(rf)
from coding.Data_Preprocessing import DataPreprocessing_melt as mf
from coding.Data_Processing import TargetVariables as tv
from sklearn.preprocessing import LabelEncoder
from scipy.stats import spearmanr
from torch.cuda.amp import autocast
encoder = LabelEncoder()
MAX_HDRS = 69 
MAX_CDI = 90  

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
class Tee(object):
    def __init__(self, *files):
        self.files = files
    def write(self, obj):
        for f in self.files:
            f.write(obj)
            f.flush()
    def flush(self):
        for f in self.files:
            f.flush()

In [4]:
log_path = f"CNN_Extractor_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
orig_stdout = sys.stdout
orig_stderr = sys.stderr
log_file = open(log_path, "w", buffering=1) 
sys.stdout = Tee(orig_stdout, log_file)
sys.stderr = Tee(orig_stderr, log_file)
start_time = time.perf_counter()
print("---- Logging started ----")
print("Log file:", log_path)

---- Logging started ----
Log file: CNN_Extractor_20260728_180344.log
Patients: 61


/workspace/app/coding/Data_Processing/TargetVariables.py:18: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_model.insert(len(df_model.columns), 'Y_Standardized_T1', None)


Processing CNN Extractor - Regression
LOPO - CNN
[CNN][Patient 0] Epoch 1/20 - train_loss=0.0271 train_monitor_loss=0.0439
[CNN][Patient 0] Epoch 2/20 - train_loss=0.0243 train_monitor_loss=0.0262
[CNN][Patient 0] Epoch 3/20 - train_loss=0.0233 train_monitor_loss=0.0234
[CNN][Patient 0] Epoch 4/20 - train_loss=0.0227 train_monitor_loss=0.0225
[CNN][Patient 0] Epoch 5/20 - train_loss=0.0225 train_monitor_loss=0.0251
[CNN][Patient 0] Epoch 6/20 - train_loss=0.0222 train_monitor_loss=0.0250
[CNN][Patient 0] Epoch 7/20 - train_loss=0.0221 train_monitor_loss=0.0247
[CNN][Patient 0] Epoch 8/20 - train_loss=0.0219 train_monitor_loss=0.0257
[CNN][Patient 0] Epoch 9/20 - train_loss=0.0218 train_monitor_loss=0.0259
[CNN][Patient 0] Epoch 10/20 - train_loss=0.0217 train_monitor_loss=0.0246
[CNN][Patient 0] Epoch 11/20 - train_loss=0.0215 train_monitor_loss=0.0275
[CNN][Patient 0] Epoch 12/20 - train_loss=0.0213 train_monitor_loss=0.0279
[CNN][Patient 0] Epoch 13/20 - train_loss=0.0213 train_monit

In [5]:
df = pd.read_pickle('/workspace/app/planilhas/df_LogMel_NotNorm.pkl')
df['Patient_ID'] = encoder.fit_transform(df['Patient_ID'])
print(f"Patients: {df['Patient_ID'].nunique()}")

In [6]:
df = tv.standardized_T1(df, MAX_HDRS, MAX_CDI)
df["Y_Standardized_T1"] = pd.to_numeric(df["Y_Standardized_T1"], errors="coerce")

In [7]:
df.drop(columns=['Questionnary3_TAS_20_T0_TOT_Score', 'Questionnary4_AQC_T0_TOT_Score', 'Questionnary5_HQ_25_T0_TOT_Score'], inplace=True)
columns_to_remove = [col for col in df.columns if col.endswith('Date') or col.endswith('F1_Score') or col.endswith('F2_Score') or col.endswith('_Days') or col.endswith('F3_Score') 
                     or col in ["Age", 'Patient_Gender', 'Randomization_Group', 'Education', 'Number_Complete_Sessions', 
                    'Questionnary3_TAS_20_T1_TOT_Score', 'Questionnary4_AQC_T1_TOT_Score','Questionnary5_HQ_25_T1_TOT_Score']
                     or col.startswith('Questionnary1') or col.startswith('Questionnary2') or col.startswith('LogMel_Pat_Speech') 
                     or col.startswith('Days_') or col.endswith('soc_Score') or col.endswith('iso_Score') or col.endswith('sup_Score')] 
df.drop(columns=columns_to_remove, inplace=True)

___

In [8]:
meta = ['Patient_ID', 'Y_Standardized_T1']
df = mf.get_logMel_per_segment_mean_3D(df, meta)

In [9]:
unique_patients = df['Patient_ID'].unique()
target = "Y_Standardized_T1"
num_epochs = 20
BS=1024
model_name = "CNN"

In [10]:
# CNN Extractor
print(f"Processing {model_name} Extractor - Regression")
results = [rf.reg_lopoCNN(p, df, "Patient_ID", "logMel_npy", target,
                           model_name, num_epochs=num_epochs, BS=BS, device="cuda")
    for p in unique_patients]
print("CNN Done")
metric_results = [
        {key: value
        for key, value in result.items()
        if key not in ["train_emb_df", "test_emb_df"]}
    for result in results]
df_results = pd.DataFrame(metric_results)
df_summary = pd.DataFrame([{
    "Model": model_name,
    "RMSE_Mean": df_results["RMSE"].mean(),
    "RMSE_Std": df_results["RMSE"].std(),
    "MSE_Mean": df_results["MSE"].mean(),
    "MAE_Mean": df_results["MAE"].mean()
}])
out_dir = "/workspace/app/planilhas"
os.makedirs(out_dir, exist_ok=True)
df_results.to_excel(os.path.join(out_dir, f"{model_name}_Extractor.xlsx"), index=False)
df_summary.to_excel(os.path.join(out_dir, f"{model_name}_Extractor_SUMMARY.xlsx"), index=False)

In [11]:
elapsed_time = time.perf_counter() - start_time
print("\n---- Experiment finished ----")
print(f"Total execution time: {elapsed_time:.2f} seconds")
print(f"Total execution time: {elapsed_time / 60:.2f} minutes")